# MMELON Fine-Tuning

Fine-tune the pretrained MMELON multi-view model (`ibm-research/biomed.sm.mv-te-84m`) for
**binary classification** of molecules (e.g. hit / non-hit) from SMILES.

**Environment:** run with the `biomed-multiview` kernel (see [INSTALL.md](../INSTALL.md)).

**Inputs:** a train CSV and a test CSV, each with a SMILES column and a 0/1 label column.

**Metrics**
- **hit@K** (hit@200, hit@100, hit@50): rank the test molecules by predicted score; hit@K is how many
  of the top K have label 1. If the test set has fewer than K molecules, all of them are counted.
- Standard classification metrics: AUROC, precision, recall, specificity and loss.

**How the model is selected:** train on the train file for up to `EPOCHS` epochs, scoring the test file after
every epoch. Epochs are ranked by `SELECT_METRICS` (default: hit@200, then hit@100, then hit@50, then AUROC to
break ties), and the best epoch is kept as the fine-tuned model. Training stops early when `PATIENCE` epochs
in a row bring no improvement, and only the best epoch's checkpoint is kept on disk.

Because the test file is used to pick the epoch, its score is slightly optimistic.
For an unbiased estimate, evaluate the final model on a separate held-out file.

For k-fold cross-validation, use [MMELON_finetuning_cv.ipynb](MMELON_finetuning_cv.ipynb) instead.

**Output** (in `aircheck/output/`, which is gitignored):
```
output/
├── base_models/biomed.sm.mv-te-84m/     # pretrained model from Hugging Face (downloaded once)
└── <DATASET>/finetuned/<RUN_NAME>_<timestamp>/
    ├── config.json                      # settings used for this run
    ├── data/                            # data_train.csv, data_val.csv (= the test file)
    ├── logs/metrics.csv                 # metrics per epoch
    ├── metrics_per_epoch.csv            # test metrics per epoch
    ├── selection.json                   # selected epoch and its test metrics
    └── best_model.ckpt                  # the fine-tuned model
```

## Steps
1. Inputs and settings
2. Prepare data
3. Download the base model
4. Data loaders
5. Build the model
6. Train and select the best epoch

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Name of this run; outputs go to output/<DATASET>/finetuned/<RUN_NAME>_<timestamp>/
RUN_NAME = "mmelon-final"

# ── Data ──────────────────────────────────────────────────────────────────────
TRAIN_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/train.parquet")
TEST_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/test.parquet")

SMILES_COL = "SMILES"
LABEL_COL = "LABEL"

In [2]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model ID

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 6
PATIENCE = 6
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

# Balance the loss between the classes: label 0 gets weight 1, label 1 gets (number of 0s) / (number of 1s),
# counted on the training data only (in CV: on each fold's own training rows). False = unweighted loss.
BALANCE_CLASSES = True

# hit@K = number of label-1 molecules among the K highest-scored test molecules
HIT_KS = [200, 100, 50]

# Model selection, in priority order: the first metric decides, later ones break ties.
# Choose from hit@K (for K in HIT_KS), "rocauc", "precision", "recall", "specificity", "loss".
SELECT_METRICS = ["hit@200", "hit@100", "hit@50", "rocauc"]

MAX_PER_CLASS = None      # e.g. 100 for a quick test run; None uses all data
NUM_WORKERS = min(24, os.cpu_count())  # data-loading processes: all CPUs, up to 24

In [3]:
import json
import shutil
from datetime import datetime

import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "aircheck" / "output"
BASE_MODEL_DIR = OUTPUT_DIR / "base_models" / BASE_MODEL.split("/")[-1]
RUN_DIR = OUTPUT_DIR / DATASET / "finetuned" / f"{RUN_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DATA_DIR = RUN_DIR / "data"

for path in (TRAIN_PATH, TEST_PATH):
    assert Path(path).exists(), f"File not found: {path}"

# Only the best epoch's checkpoint (~0.35 GB) is kept, overwritten when a later epoch is better
needed_gb = 0.35 * 2
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > needed_gb + 2, f"Need ~{needed_gb:.0f} GB free for checkpoints, only {free_gb:.0f} GB available"

RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)

config = dict(
    train_path=str(TRAIN_PATH), test_path=str(TEST_PATH),
    smiles_col=SMILES_COL, label_col=LABEL_COL, base_model=BASE_MODEL,
    epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, seed=SEED, hit_ks=HIT_KS, select_metrics=SELECT_METRICS, balance_classes=BALANCE_CLASSES,
    max_per_class=MAX_PER_CLASS,
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Repo root  : {REPO_ROOT}")
print(f"Base model : {BASE_MODEL_DIR}")
print(f"Run folder : {RUN_DIR}")
print(f"Disk       : ~{needed_gb:.1f} GB needed during training, {free_gb:.0f} GB free")
print(f"Device     : {'GPU' if torch.cuda.is_available() else 'CPU (training will be slow)'}")

Repo root  : ~/repos/biomed-multi-view
Base model : ~/repos/biomed-multi-view/aircheck/output/base_models/biomed.sm.mv-te-84m
Run folder : ~/repos/biomed-multi-view/aircheck/output/sample-data/finetuned/mmelon-final_20261020_162634
Disk       : ~0.7 GB needed during training, 37 GB free
Device     : GPU


## 2. Prepare data

The MMELON data pipeline reads `data_train.csv` and `data_val.csv` from one folder, with columns
`smiles` and `label`, and scores `data_val.csv` after every epoch. This step copies the train file to
`data_train.csv` and the **test file to `data_val.csv`** in the run's `data/` folder.

In [4]:
import pandas as pd

import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)


def load_split(path: Path) -> pd.DataFrame:
    df = read_table(path, [SMILES_COL, LABEL_COL]).rename(columns={SMILES_COL: "smiles", LABEL_COL: "label"})
    n_before = len(df)
    df = df.dropna()
    df = df[df["smiles"].astype(str).str.strip() != ""]
    if len(df) < n_before:
        print(f"{path.name}: dropped {n_before - len(df)} rows with empty SMILES or label")
    df["label"] = df["label"].astype(int)
    assert set(df["label"].unique()) <= {0, 1}, f"{path} labels must be 0/1"
    return df.reset_index(drop=True)


def limit_per_class(df: pd.DataFrame, n: int) -> pd.DataFrame:
    parts = [g.sample(n=min(n, len(g)), random_state=SEED) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)


def class_weights_for(labels: pd.Series):
    # Loss weights [[label 0, label 1]] from the training labels, or None (unweighted) when BALANCE_CLASSES is off
    if not BALANCE_CLASSES:
        return None
    n_neg, n_pos = int((labels == 0).sum()), int((labels == 1).sum())
    assert n_neg > 0 and n_pos > 0, "training data needs both labels to balance the classes"
    return [[1.0, round(n_neg / n_pos, 4)]]


summary = []
for name, path, out_name in [("train", TRAIN_PATH, "data_train.csv"), ("test", TEST_PATH, "data_val.csv")]:
    df = load_split(path)
    if MAX_PER_CLASS is not None:
        df = limit_per_class(df, MAX_PER_CLASS)
    df.to_csv(RUN_DATA_DIR / out_name, index=False)
    if name == "train":
        CLASS_WEIGHTS = class_weights_for(df["label"])
    summary.append(dict(split=name, samples=len(df), positives=int(df["label"].sum()),
                        positive_pct=round(100 * df["label"].mean(), 2)))

config["class_weights"] = CLASS_WEIGHTS
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
print(f"Class weights [[label 0, label 1]]: {CLASS_WEIGHTS}")
pd.DataFrame(summary)

Class weights [[label 0, label 1]]: [[1.0, 45.2489]]


,split,samples,positives,positive_pct
0,train,10221,221,2.16
1,test,1031,31,3.01


## 3. Download the base model

Downloaded once into `output/base_models/`; later runs reuse it.

In [5]:
from huggingface_hub import snapshot_download

if (BASE_MODEL_DIR / "model.safetensors").exists():
    print(f"Using existing base model: {BASE_MODEL_DIR}")
else:
    print(f"Downloading {BASE_MODEL} ...")
    snapshot_download(repo_id=BASE_MODEL, local_dir=BASE_MODEL_DIR)
    print(f"Saved to: {BASE_MODEL_DIR}")

Using existing base model: ~/repos/biomed-multi-view/aircheck/output/base_models/biomed.sm.mv-te-84m


## 4. Data loaders

Each SMILES is converted into the three MMELON views: text tokens, a 2D image and a molecular graph.

In [6]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

pl.seed_everything(SEED)

dataset_args = {
    "task_type": TaskType.CLASSIFICATION,
    "num_tasks": 1,
    "modalities": ["TEXT_MODEL", "IMAGE_MODEL", "GRAPH_2D_MODEL"],
    "smiles_col": "smiles",
    "label_cols": ["label"],
    "split_col": None,
}


def make_loader(stage: str, shuffle: bool) -> DataLoader:
    dataset = MultiModalFinetuneDataPipeline(
        data_dir=str(RUN_DATA_DIR), dataset_args=dataset_args, stage=stage
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)


train_loader = make_loader("train", shuffle=True)
test_loader = make_loader("val", shuffle=False)  # "val" stage reads data_val.csv = the test file

for name, loader in [("train", train_loader), ("test", test_loader)]:
    print(f"{name:5s}: {len(loader.dataset):6,} samples, {len(loader):5,} batches")

2026-10-20 16:26:38,431 - rdkit - INFO - instance-shay-ml-01:139802562242368:0:0 - Enabling RDKit 2026.09.1 jupyter extensions


Seed set to 42


2026-10-20 16:26:38,621 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Working on task TaskType.CLASSIFICATION


2026-10-20 16:26:38,622 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


2026-10-20 16:26:40,355 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Working on task TaskType.CLASSIFICATION


2026-10-20 16:26:40,356 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


train: 10,221 samples,   639 batches
test :  1,031 samples,    65 batches


## 5. Build the model

Creates the fine-tuning model (three encoders + attention fusion + MLP prediction head) and loads the
pretrained encoder weights. The prediction head starts from random weights.
The model is extended to also log hit@K after every epoch.

In [7]:
class HitAtKFineTuneModule(FineTuneLightningModule):
    # Adds hit@K to the scores logged after every epoch (as val_hit@K)

    def on_validation_epoch_start(self):
        super().on_validation_epoch_start()
        self._val_scores, self._val_labels = [], []

    def validation_step(self, batch, batch_idx):
        loss, y, y_hat = self._shared_step(batch, "val")
        self._val_scores.append(y_hat.detach().float().reshape(-1).cpu())
        self._val_labels.append(y.detach().reshape(-1).cpu())
        return loss

    def on_validation_epoch_end(self):
        super().on_validation_epoch_end()
        scores, labels = torch.cat(self._val_scores), torch.cat(self._val_labels)
        top = torch.argsort(scores, descending=True)
        for k in HIT_KS:
            self.log(f"val_hit@{k}", float((labels[top[:k]] == 1).sum()))


FUSION_STRATEGY = LateFusionStrategy.ATTENTIONAL

lightning_module = HitAtKFineTuneModule(
    base_model_class="bmfm_sm.predictive.modules.smmv_model.SmallMoleculeMultiView",
    model_params={
        "agg_arch": FUSION_STRATEGY.value[0],
        "agg_gate_input": FUSION_STRATEGY.value[1],
        "agg_weight_freeze": FUSION_STRATEGY.value[2],
        "inference_mode": False,
    },
    task_type="classification",
    num_tasks=1,
    checkpoint_path=None,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    finetuning_args={
        "weight_freeze": "unfrozen",
        "initialization": "default",
        "head_arch": "mlp",
        "use_norm": True,
        "head_dropout": 0.2,
        "class_weights": CLASS_WEIGHTS,
    },
)

pretrained_model = SmallMoleculeMultiViewModel.from_pretrained(
    FUSION_STRATEGY, model_path=str(BASE_MODEL_DIR), huggingface=True
)
result = lightning_module.model.load_state_dict(pretrained_model.state_dict(), strict=False)
del pretrained_model

# strict=False would silently skip mismatched weights, so check explicitly
assert not result.missing_keys and not result.unexpected_keys, (
    f"Pretrained weights did not match: missing={result.missing_keys[:5]}, "
    f"unexpected={result.unexpected_keys[:5]}"
)
print("Pretrained weights loaded (all keys matched).")

2026-10-20 16:26:41,715 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Initializing model with params {'agg_arch': 'coeff_mlp', 'agg_gate_input': 'coeff_mlp', 'agg_weight_freeze': None, 'inference_mode': False}


2026-10-20 16:26:43,040 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Using coeff_mlp architecture for aggregator


2026-10-20 16:26:43,041 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - dim_list [512, 512, 768] for aggregator


2026-10-20 16:26:43,069 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Initialized the model with random weights as checkpoint_path was specified as None


2026-10-20 16:26:43,070 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Task type is classification


...

2026-10-20 16:26:43,089 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Loading checkpoint via HuggingFace Hub from provided path ~/repos/biomed-multi-view/aircheck/output/base_models/biomed.sm.mv-te-84m


2026-10-20 16:26:44,408 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - Using coeff_mlp architecture for aggregator


2026-10-20 16:26:44,409 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - dim_list [512, 512, 768] for aggregator


2026-10-20 16:26:44,515 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - in train False setting deterministic_eval = True


Loading weights from local directory
Pretrained weights loaded (all keys matched).


## 6. Train and select the best epoch

After every epoch the model is scored on the test file and ranked by `SELECT_METRICS` (the first metric decides,
later ones break ties). The best epoch so far is saved as `best_model.ckpt`, replacing the previous best, and
training stops early when `PATIENCE` epochs in a row bring no improvement.
Metrics prefixed `val_` in the logs are scores on the **test** file.

In [8]:
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import CSVLogger

allowed = [f"hit@{k}" for k in HIT_KS] + ["rocauc", "precision", "recall", "specificity", "loss"]
assert set(SELECT_METRICS) <= set(allowed), f"SELECT_METRICS must come from {allowed}"


class KeepBestEpoch(Callback):
    # After each epoch: save it if it is the best so far by SELECT_METRICS, and stop after PATIENCE epochs
    # without improvement. Only one checkpoint is kept on disk.

    def __init__(self, path: Path):
        self.path, self.best_key, self.best_epoch, self.wait = path, None, None, 0

    def on_validation_end(self, trainer, pl_module):
        logged = trainer.callback_metrics
        # Larger is better; loss is negated so a lower loss ranks higher. Ties keep the earlier epoch.
        key = tuple(float(logged[f"val_{m}"]) * (-1 if m == "loss" else 1) for m in SELECT_METRICS)
        if self.best_key is None or key > self.best_key:
            self.best_key, self.best_epoch, self.wait = key, trainer.current_epoch, 0
            trainer.save_checkpoint(self.path, weights_only=True)
        else:
            self.wait += 1
            if self.wait >= PATIENCE:
                print(f"Stopping early after epoch {trainer.current_epoch}: no improvement for {PATIENCE} epochs")
                trainer.should_stop = True


keeper = KeepBestEpoch(RUN_DIR / "best_model.ckpt")
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=[keeper],
    logger=CSVLogger(save_dir=RUN_DIR, name="logs", version=""),
    enable_checkpointing=False,
    default_root_dir=RUN_DIR,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    num_sanity_val_steps=0,
    enable_model_summary=True,
)

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [9]:
%%time
trainer.fit(lightning_module, train_loader, test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ SmallMoleculeMultiView  │ 84.1 M │ train │     0 │
│ 1 │ pred_head │ MultiTaskPredictionHead │  461 K │ train │     0 │
│ 2 │ criterion │ BCEWithLogitsLoss       │      0 │ train │     0 │
└───┴───────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 84.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 84.6 M                                                                                               
Total estimated model params size (MB): 338.207                                                                    
Modules in train mode: 510                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

2026-10-20 16:30:07,960 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - in train False setting deterministic_eval = True


2026-10-20 16:33:40,843 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - in train False setting deterministic_eval = True


2026-10-20 16:37:11,108 - root - INFO - instance-shay-ml-01:139802562242368:0:0 - in train False setting deterministic_eval = True


In [10]:
# Test metrics per epoch
logged = pd.read_csv(RUN_DIR / "logs" / "metrics.csv")
metrics = logged.dropna(subset=["val_loss"])
metric_cols = [c for c in metrics.columns if c.startswith("val_")]
metrics = metrics[["epoch"] + metric_cols].rename(columns=lambda c: c.removeprefix("val_"))
metrics = metrics.groupby("epoch", as_index=False).first()  # one row per epoch
metrics["epoch"] = metrics["epoch"].astype(int)

# hit@K first (in HIT_KS order), then the standard metrics
hits = [f"hit@{k}" for k in HIT_KS]
metric_names = hits + [c for c in metrics.columns if c not in hits and c != "epoch"]
metrics = metrics[["epoch"] + metric_names]
metrics.to_csv(RUN_DIR / "metrics_per_epoch.csv", index=False)

# Check: ranking all epochs afterwards picks the same epoch as the choice made during training
ranked = metrics.sort_values(SELECT_METRICS, ascending=[m == "loss" for m in SELECT_METRICS], kind="mergesort")
best_epoch = keeper.best_epoch
assert int(ranked.iloc[0]["epoch"]) == best_epoch, "epoch chosen during training differs from the ranking"
best_metrics = metrics.set_index("epoch").loc[best_epoch]
best_path = RUN_DIR / "best_model.ckpt"

epochs_run = len(metrics)
selection = dict(
    select_metrics=SELECT_METRICS,
    best_epoch=best_epoch,
    epochs_trained=epochs_run,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    stopped_early=epochs_run < EPOCHS,
    test_metrics={m: float(best_metrics[m]) for m in metric_names},
    model=str(best_path),
)
(RUN_DIR / "selection.json").write_text(json.dumps(selection, indent=2))

print(f"Trained {epochs_run} of up to {EPOCHS} epochs{' (stopped early)' if epochs_run < EPOCHS else ''}")
print(f"Selected epoch {best_epoch}:")
for m in SELECT_METRICS:
    print(f"  {m:12s} {best_metrics[m]:.4f}")
print(f"Fine-tuned model: {best_path}\n")
display(metrics.round(4))

Trained 6 of up to 6 epochs
Selected epoch 3:
  hit@200      23.0000
  hit@100      13.0000
  hit@50       10.0000
  rocauc       0.8264
Fine-tuned model: ~/repos/biomed-multi-view/aircheck/output/sample-data/finetuned/mmelon-final_20261020_162634/best_model.ckpt



,epoch,hit@200,hit@100,hit@50,loss,precision,recall,rocauc,specificity
0,0,22.0,15.0,10.0,1.5001,0.1707,0.2258,0.8280,0.966
1,1,18.0,13.0,8.0,1.3720,0.0441,0.9355,0.7821,0.371
2,2,21.0,16.0,12.0,1.3075,0.1143,0.6452,0.8449,0.845
3,3,23.0,13.0,10.0,1.3060,0.1036,0.7419,0.8264,0.801
4,4,22.0,16.0,12.0,1.2378,0.0852,0.7419,0.8291,0.753
5,5,21.0,15.0,13.0,1.2404,0.1223,0.5484,0.8340,0.878


In [11]:
# Files produced by this run
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(RUN_DIR)):60s} {path.stat().st_size / 1e6:8.2f} MB")

best_model.ckpt                                                338.59 MB
config.json                                                      0.00 MB
data/data_train.csv                                              0.44 MB
data/data_val.csv                                                0.04 MB
logs/hparams.yaml                                                0.00 MB
logs/metrics.csv                                                 0.12 MB
metrics_per_epoch.csv                                            0.00 MB
selection.json                                                   0.00 MB
